# Review tables: preprocessing, grids, selected features, and H/C/FIM ablations

This notebook reconstructs the four additions requested during editorial review. It does not refit any classifier: it reads the executed validation artifacts, checks their grain, and exports publication-ready tables.

## tl;dr

- The released derivatives used by the manuscript were filtered at 0.5–45 Hz, rereferenced to A1–A2, corrected with ASR, decomposed with RunICA, and cleaned by ICLabel rejection of eye and jaw components.
- The original nested classifier grids are made explicit below; retained feature count was set separately from the classifier grid.
- Fold-wise HCFIM selection is summarized from the duplicate-aware repeated 10×5 analysis at matched k=30.
- All seven nonempty H/C/FIM coordinate subsets are tabulated for both selected temporal geometries.

## Context & Methods

### Key assumptions

The selected-feature table uses the duplicate-aware repeated-validation artifact because deterministic aliases can otherwise inflate apparent selection frequency. For each region–band–coordinate profile, the table reports the most frequently selected concrete transform/statistic. Frequencies use 50 outer fold models (10 repetitions × 5 folds) as denominator.

The preprocessing description is transcribed from the primary ds004504 v1.0.6 dataset documentation: https://openneuro.org/datasets/ds004504/versions/1.0.6

In [ ]:
from pathlib import Path
import os
import pandas as pd

def resolve_project_root() -> Path:
    """Locate the repository without relying on a machine-specific path."""
    env_root = os.environ.get("HCFIM_PROJECT_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
        raise FileNotFoundError(
            "HCFIM_PROJECT_ROOT does not point to the HCFIM-eeg-research repository."
        )

    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository. Run the notebook from within the repository "
        "or set HCFIM_PROJECT_ROOT."
    )

REPO = resolve_project_root()
RESULTS = REPO / 'results'
OUT = RESULTS / 'reviewer_revision_outputs'
OUT.mkdir(parents=True, exist_ok=True)

selected_path = RESULTS / 'feature_duplicate_audit' / 'deduplicated_repeated_selected_features.csv'
ablation_path = RESULTS / 'confirmatory_sensitivity' / 'hcfim_dimensional_ablation.csv'
assert selected_path.exists(), selected_path
assert ablation_path.exists(), ablation_path
print('Inputs verified')

## Data

### 1. Published derivative preprocessing

In [ ]:
preprocessing = pd.DataFrame([
    {'Stage': 'Band-pass filtering', 'Published operation': 'Butterworth 0.5–45 Hz'},
    {'Stage': 'Rereferencing', 'Published operation': 'A1–A2 mastoid reference'},
    {'Stage': 'Artifact Subspace Reconstruction', 'Published operation': 'ASR; 0.5-s window, cutoff 17 SD'},
    {'Stage': 'Independent component analysis', 'Published operation': 'RunICA; 19 signals mapped to 19 components'},
    {'Stage': 'Component rejection', 'Published operation': 'ICLabel; automatic rejection of eye- and jaw-artifact components'},
])
preprocessing.to_csv(OUT / 'published_derivative_preprocessing_summary.csv', index=False)
preprocessing

### 2. Original nested classifier grids

In [ ]:
grids = pd.DataFrame([
    ['Elastic-net logistic regression', 'saga; balanced class weights; max_iter=5000', 'C={0.05, 0.1, 0.5, 1.0}; l1_ratio={0.15, 0.5, 0.85}'],
    ['RBF SVM', 'probability=True; balanced class weights', 'C={0.1, 1, 10}; gamma={scale, 0.01, 0.1}'],
    ['Random Forest', 'balanced class weights', 'n_estimators=200; max_depth={2, 3, None}'],
    ['Extra Trees', 'balanced class weights', 'n_estimators=300; max_depth={2, 3, None}'],
    ['Gradient Boosting', 'remaining settings at defaults', 'n_estimators={50, 100}; learning_rate={0.03, 0.1}; max_depth={1, 2}'],
    ['Small MLP', 'max_iter=2000', 'alpha={0.001, 0.01, 0.1}; hidden_layer_sizes={(8,), (12,)}'],
], columns=['Classifier', 'Fixed settings', 'Inner-fold grid'])
grids.to_csv(OUT / 'classifier_hyperparameter_grids.csv', index=False)
grids

## Results

### 3. Most frequently selected HCFIM profiles

In [ ]:
selected = pd.read_csv(selected_path)
selected = selected[(selected['feature_set'] == 'HCFIM') & (selected['k'] == 30)].copy()
fold_counts = selected.groupby('task')[['repeat', 'fold']].apply(lambda x: x.drop_duplicates().shape[0])
assert (fold_counts == 50).all(), fold_counts

counts = selected.groupby(['task', 'feature']).size().rename('selected_folds').reset_index()
counts['selection_frequency'] = counts.apply(lambda r: r['selected_folds'] / fold_counts[r['task']], axis=1)
parts = counts['feature'].str.split('__', expand=True)
parts.columns = ['prefix', 'transform', 'distance', 'region', 'band', 'coordinate', 'summary']
counts = pd.concat([counts, parts[['transform', 'distance', 'region', 'band', 'coordinate', 'summary']]], axis=1)
idx = counts.groupby(['task', 'region', 'band', 'coordinate'])['selection_frequency'].idxmax()
profiles = counts.loc[idx].sort_values(['task', 'selection_frequency'], ascending=[True, False]).copy()
profiles['comparison'] = profiles['task'].map({
    'control_vs_alzheimer': 'Control vs AD',
    'control_vs_frontotemporal_dementia': 'Control vs FTD',
})
profiles['selected_outer_folds'] = profiles['selected_folds'].astype(int).astype(str) + '/50'
profiles['frequency_percent'] = (100 * profiles['selection_frequency']).round(0).astype(int)
top_profiles = profiles.groupby('task', group_keys=False).head(6).copy()
top_profiles['Section 3.3 concordance'] = ''
top_profiles.loc[top_profiles['task'].eq('control_vs_alzheimer'), 'Section 3.3 concordance'] = (
    'Temporal-alpha FIM agrees; occipital-alpha FIM was equally frequent'
)
top_profiles.loc[top_profiles['task'].eq('control_vs_frontotemporal_dementia'), 'Section 3.3 concordance'] = (
    'Broadband FIM agrees; frontal profiles were more frequent than temporal'
)
cols = ['comparison', 'region', 'band', 'coordinate', 'transform', 'summary', 'selected_outer_folds', 'frequency_percent', 'Section 3.3 concordance']
top_profiles[cols].to_csv(OUT / 'top_hcfim_selected_profiles_k30.csv', index=False)
profiles.to_csv(OUT / 'all_hcfim_selected_profiles_k30.csv', index=False)
top_profiles[cols]

The AD selection profile is concordant with the temporal-alpha finding but not uniquely localized: occipital-alpha FIM summaries were selected just as often. The FTD profile agrees at the broadband/FIM level, while frontal broadband features were more stable than the temporal geometry highlighted by the whole-cohort ranking.

### 4. Seven nonempty H/C/FIM coordinate subsets

In [ ]:
ablations = pd.read_csv(ablation_path).copy()
ablations['Comparison'] = ablations['groups'].map({
    'control | alzheimer': 'Control vs AD',
    'control | frontotemporal_dementia': 'Control vs FTD',
})
ablation_table = ablations[['Comparison', 'geometry', 'coordinates', 'dimension', 'roc_auc', 'balanced_accuracy', 'f1']].copy()
ablation_table.columns = ['Comparison', 'Selected geometry', 'Coordinates', 'Dimension', 'ROC AUC', 'Balanced accuracy', 'F1']
for col in ['ROC AUC', 'Balanced accuracy', 'F1']:
    ablation_table[col] = ablation_table[col].round(3)
ablation_table.to_csv(OUT / 'seven_hcfim_coordinate_ablations.csv', index=False)
assert ablation_table.groupby('Comparison').size().eq(7).all()
ablation_table

## Takeaways

The complete ablation table confirms that the three-coordinate representation was not uniformly best. C+FIM had the highest AD point estimate (0.740), whereas FIM alone had the highest FTD point estimate (0.710). Fold-wise feature selection supports temporal-alpha FIM as an AD-relevant profile, but it also shows equally frequent occipital-alpha FIM selection. For FTD, broadband information recurs, but the most stable selected profiles are frontal rather than uniquely temporal.